# EVOLVE — YOLO Evaluation Notebook

This notebook is dedicated to the **evaluation and qualitative analysis** of
the YOLO-based object detection model trained on the EVOLVE dataset.

The objectives are:
- quantitative evaluation (mAP, precision, recall)
- per-class performance analysis
- qualitative inspection of detections
- identification of common failure modes

All dataset design choices and annotation rules are documented in
`evolve_workbook.qmd`.

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
from pathlib import Path
import random
import pandas as pd
from PIL import Image

## Load trained model

The model corresponds to the best weights obtained during training.

In [ ]:
model = YOLO("runs/evolve_yolo/weights/best.pt")

## Quantitative evaluation

We evaluate the model on the unseen test split to obtain a final measure of performance.

In [ ]:
metrics = model.val(
    data="data/yolo/dataset.yaml",
    split='test'
)

metrics

## Per-class Average Precision (AP)

Beyond global mAP values, we inspect detection performance **per class**
to better understand model behavior under extreme visual conditions.

In [ ]:
names = metrics.names
ap50 = metrics.box.ap50
ap = metrics.box.ap

per_class_ap = {
    names[i]: {
        "AP@50": float(ap50[i]),
        "AP@50:95": float(ap[i])
    }
    for i in range(len(names))
}

per_class_ap

In [ ]:
df_ap = pd.DataFrame(per_class_ap).T
df_ap

### Interpretation of per-class performance

Several trends emerge from the per-class AP values:

- **Higher AP** for large, static objects such as `amp` and `drums` due to more consistent geometric features.
- **Lower AP** for small or thin objects like `micro`, which are easily lost in high-frequency noise.
- **Coarse but semantically meaningful performance** for collective patterns (`mosh_pit`, `hands_raised`), where boundaries are inherently fuzzy.

## Qualitative evaluation on test images

Visual inspection allows us to identify success cases (robustness to lighting) and common failure modes (scale and occlusion errors).

In [ ]:
test_images = list(Path("data/yolo/images/test").glob("*.jpg"))
sample_images = random.sample(test_images, k=4)
sample_images

In [ ]:
results = model.predict(
    source=[str(p) for p in sample_images],
    conf=0.25,
    save=False
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

for ax, res in zip(axes.flatten(), results):
    img = Image.fromarray(res.plot()[:, :, ::-1])
    ax.imshow(img)
    ax.axis("off")

plt.tight_layout()

### Qualitative observations

- **Success:** Robust detection of stage gear under saturated colored lights.
- **Failure:** Localization errors in dense crowd zones and missed small microphones.
- **Insight:** The model relies on global scene context when local features are camouflaged by deep shadows.

## Discussion: object scale and annotation granularity

Standard IoU metrics may underestimate the semantic success for classes like `mosh_pit`, where the target is a collective behavior rather than a sharp physical object. Qualitative inspection remains essential for these classes.

## End of evaluation notebook